In [1]:
import pandas as pd
import numpy as np
import sqlite3

In [2]:
nav_hist = pd.read_csv("../data/raw/02_nav_history.csv")

invest_transactions = pd.read_csv("../data/raw/08_investor_transactions.csv")

scheme_performance = pd.read_csv("../data/raw/07_scheme_performance.csv")

In [3]:
print(nav_hist.head())

print(nav_hist.info())

   amfi_code        date      nav
0     119551  2022-01-03  54.3856
1     119551  2022-01-04  54.3474
2     119551  2022-01-05  54.6869
3     119551  2022-01-06  55.4550
4     119551  2022-01-07  55.3692
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   amfi_code  46000 non-null  int64  
 1   date       46000 non-null  object 
 2   nav        46000 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 1.1+ MB
None


In [4]:
print(nav_hist.columns)

Index(['amfi_code', 'date', 'nav'], dtype='object')


In [5]:
nav_hist["date"] = pd.to_datetime(nav_hist["date"])

In [6]:
nav_hist = nav_hist.sort_values(
    by=["amfi_code", "date"]
)

In [7]:
nav_hist = nav_hist.drop_duplicates()

In [8]:
print(nav_hist.columns)

Index(['amfi_code', 'date', 'nav'], dtype='object')


In [9]:
nav_hist = nav_hist[
    nav_hist["nav"] > 0
]

In [10]:
nav_hist["nav"] = nav_hist["nav"].ffill()

In [11]:
nav_hist.to_csv(
    "../data/processed/cleaned_nav_history.csv",
    index=False
)

In [12]:
print(invest_transactions.columns)

Index(['investor_id', 'transaction_date', 'amfi_code', 'transaction_type',
       'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender',
       'annual_income_lakh', 'payment_mode', 'kyc_status'],
      dtype='object')


In [13]:
invest_transactions = invest_transactions.drop_duplicates()

In [15]:
invest_transactions["transaction_date"] = pd.to_datetime(
    invest_transactions["transaction_date"]
)

In [16]:
invest_transactions = invest_transactions[
    invest_transactions["amount_inr"] > 0
]

In [17]:
invest_transactions["transaction_type"] = (
    invest_transactions["transaction_type"]
    .str.upper()
)

In [18]:
invest_transactions.to_csv(
    "../data/processed/cleaned_transactions.csv",
    index=False
)

In [19]:
print(scheme_performance.columns)

Index(['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan',
       'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct',
       'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio',
       'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct',
       'morningstar_rating', 'risk_grade'],
      dtype='object')


In [20]:
scheme_performance = scheme_performance.drop_duplicates()

In [23]:
scheme_performance = scheme_performance[
    scheme_performance["return_1yr_pct"] >= 0
]

In [24]:
scheme_performance.to_csv(
    "../data/processed/cleaned_performance.csv",
    index=False
)

In [25]:
import sqlite3

In [26]:
conn = sqlite3.connect(
    "../data/db/bluestock_mf.db"
)

print("Database Connected")

Database Connected


In [27]:
cleaned_nav = pd.read_csv(
    "../data/processed/cleaned_nav_history.csv"
)

cleaned_transactions = pd.read_csv(
    "../data/processed/cleaned_transactions.csv"
)

cleaned_performance = pd.read_csv(
    "../data/processed/cleaned_performance.csv"
)

In [28]:
cleaned_nav.to_sql(
    "fact_nav",
    conn,
    if_exists="replace",
    index=False
)

cleaned_transactions.to_sql(
    "fact_transactions",
    conn,
    if_exists="replace",
    index=False
)

cleaned_performance.to_sql(
    "fact_performance",
    conn,
    if_exists="replace",
    index=False
)

print("Tables Loaded Successfully")

Tables Loaded Successfully


In [29]:
query = "SELECT * FROM fact_nav LIMIT 5"

result = pd.read_sql(query, conn)

print(result)

   amfi_code        date       nav
0     100016  2022-01-03  520.4608
1     100016  2022-01-04  515.0971
2     100016  2022-01-05  521.7239
3     100016  2022-01-06  515.7880
4     100016  2022-01-07  515.1639
